In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,udf
from pyspark.sql.types import StringType, IntegerType
import time

In [2]:
spark = SparkSession.builder.appName("OptimizationDemo").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 06:00:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.range(0, 10000000)
df.show(5)

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+
only showing top 5 rows



In [4]:
optimized_df = df.filter(col("id") > 500) \
                 .filter(col("id") < 5000000) \
                 .select("id")

26/06/13 06:00:32 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [5]:
optimized_df.explain()

== Physical Plan ==
*(1) Filter ((id#0L > 500) AND (id#0L < 5000000))
+- *(1) Range (0, 10000000, step=1, splits=2)




In [6]:
start_time = time.time()

count_uncached = optimized_df.count()

end_time = time.time()

print(
    f"1. Optimized Execution | Count: {count_uncached} | Time: {round(end_time - start_time, 4)} seconds"
)

1. Optimized Execution | Count: 4999499 | Time: 0.6301 seconds


In [7]:
optimized_df.cache()

DataFrame[id: bigint]

In [8]:
optimized_df.count()

4999499

In [9]:
start_time = time.time()

count_cached = optimized_df.count()

end_time = time.time()

print(
    f"2. Cached Execution | Count: {count_cached} | Time: {round(end_time - start_time, 4)} seconds"
)

2. Cached Execution | Count: 4999499 | Time: 0.19 seconds


In [10]:
optimized_df.unpersist()

DataFrame[id: bigint]